In [ ]:
# This setup is only needed in Colab, which opens the notebook without cloning
# the repo it came from. If you already have the repo, skip this cell.
!git clone https://github.com/yiftachbeer/notebook-to-production/ -b main
%cd notebook-to-production
import sys
sys.path.insert(0, ".")

# Depth + Segmentation -> Colored 3D Point Cloud

Run a monocular depth model and a semantic segmentation model on a single
driving image, then back-project every pixel into a 3D point cloud colored by
its semantic class, using the camera's real calibration.


In [ ]:
!pip install -q -U transformers huggingface_hub

In [ ]:
import numpy as np
import torch
from PIL import Image
from transformers import pipeline
import matplotlib.pyplot as plt
import plotly.graph_objects as go

DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Base-hf"
SEG_MODEL = "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
N_POINTS = 60000

DEVICE = 0 if torch.cuda.is_available() else -1
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
print("CUDA:", torch.cuda.is_available())

CITYSCAPES_COLORS = {
    "road": (128, 64, 128), "sidewalk": (244, 35, 232), "building": (70, 70, 70),
    "wall": (102, 102, 156), "fence": (190, 153, 153), "pole": (153, 153, 153),
    "traffic light": (250, 170, 30), "traffic sign": (220, 220, 0),
    "vegetation": (107, 142, 35), "terrain": (152, 251, 152), "sky": (70, 130, 180),
    "person": (220, 20, 60), "rider": (255, 0, 0), "car": (0, 0, 142),
    "truck": (0, 0, 70), "bus": (0, 60, 100), "train": (0, 80, 100),
    "motorcycle": (0, 0, 230), "bicycle": (119, 11, 32),
}

## Load the data

In [ ]:
!wget https://stereo-vision.s3.eu-west-3.amazonaws.com/stereo_data.zip && jar xf stereo_data.zip

In [ ]:
import os
os.chdir("stereo_data")

In [ ]:
IMAGE_PATH = "/content/notebook-to-production/stereo_data/left/000009.png"
CALIB_PATH = "/content/notebook-to-production/stereo_data/calib/000009.txt"

image = Image.open(IMAGE_PATH).convert("RGB")
W, H = image.size

# real camera intrinsics for the left color camera (P2 in the KITTI calib)
for line in open(CALIB_PATH):
    if line.startswith("P2:"):
        P2 = np.array(line.split()[1:], dtype=np.float32).reshape(3, 4)
fx, fy = P2[0, 0], P2[1, 1]
cx, cy = P2[0, 2], P2[1, 2]
print("fx, fy, cx, cy =", fx, fy, cx, cy)

plt.figure(figsize=(12, 4))
plt.imshow(image)
plt.axis("off")
plt.title("input")
plt.show()

## Step 1 - Depth

In [ ]:
depth_pipe = pipeline("depth-estimation", model=DEPTH_MODEL, device=DEVICE, torch_dtype=DTYPE)
pred = depth_pipe(image)["predicted_depth"].squeeze().float().cpu().numpy()
if pred.shape != (H, W):
    pred = np.array(Image.fromarray(pred).resize((W, H), Image.BILINEAR))
depth_m = pred  # metric depth, in meters

plt.figure(figsize=(12, 4))
plt.imshow(depth_m, cmap="magma")
plt.axis("off")
plt.title("depth (m)")
plt.colorbar(fraction=0.025)
plt.show()

## Step 2 - Segmentation

In [ ]:
seg_pipe = pipeline("image-segmentation", model=SEG_MODEL, device=DEVICE, torch_dtype=DTYPE)
seg_out = seg_pipe(image)

seg_color = np.zeros((H, W, 3), dtype=np.uint8)
for seg in seg_out:
    mask = np.array(seg["mask"]) > 0
    seg_color[mask] = CITYSCAPES_COLORS.get(seg["label"], (0, 0, 0))

plt.figure(figsize=(12, 4))
plt.imshow(seg_color)
plt.axis("off")
plt.title("segmentation")
plt.show()

## Step 3 - 3D Segmentation

In [ ]:
# pinhole back-projection with the real intrinsics; depth and segmentation are
# both at native resolution, so they line up pixel-for-pixel
u, v = np.meshgrid(np.arange(W), np.arange(H))
Z = depth_m
X = (u - cx) * Z / fx
Y = (v - cy) * Z / fy

points = np.stack([X, Y, Z], axis=-1).reshape(-1, 3)
colors = seg_color.reshape(-1, 3)

idx = np.random.choice(len(points), min(N_POINTS, len(points)), replace=False)
points, colors = points[idx], colors[idx]
print(points.shape)

In [ ]:
rgb = ["rgb(%d,%d,%d)" % (r, g, b) for r, g, b in colors]

fig = go.Figure(go.Scatter3d(
    x=points[:, 0], y=-points[:, 1], z=points[:, 2],
    mode="markers",
    marker=dict(size=1.5, color=rgb, opacity=1.0),
))
fig.update_layout(
    template="plotly_dark",
    width=1000, height=650,
    margin=dict(l=0, r=0, t=0, b=0),
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        camera=dict(eye=dict(x=0.0, y=0.35, z=-1.7), up=dict(x=0, y=1, z=0)),
    ),
)
fig.show()